# Qualitative coding for a batch of interviews: de-identification, thematic analysis, and quote provenance tracing

Qualitative research asks "how do people narrate their experiences," not "how large is this number." The most common approach is **reflexive thematic analysis** (Braun & Clarke 2006/2019): partition interview text into segments, apply "code" labels to each segment, aggregate similar codes into higher-order "themes," and support each theme with verbatim quotes. Unlike regression, it has no statistical significance threshold; credibility derives elsewhere—**every conclusion can be traced back to a specific respondent's exact words**.

This pipeline has two non-negotiable gates. The first is **ethics**: interviews almost always contain personally identifiable information (PII)—names, email addresses, phone numbers—which must be de-identified before coding or sharing. De-identification must preserve a reversible crosswalk so that authorized users can re-identify if needed. The second is **auditability**: thematic analysis is criticized for "finding what the researcher wants to find." The remedy is to anchor each quote with a precise provenance stamp (which document, which character range), allowing verbatim spot-check of the original text afterwards—this step is most error-prone because de-identification changes text length and shifts character offsets. We will deliberately encounter this pitfall and repair it.

The full pipeline runs: **load corpus → de-identify → code themes → trace quotes → reflexive memo → theme map**. We implement it using `socialverse`, a library for social-science analysis. Each step reads from and writes to a shared research state, automatically recording a reproducible audit trail—you will see it at the end. The methodological framework is Braun & Clarke's reflexive thematic analysis; the workflow mirrors **CAQDAS** software (NVivo / ATLAS.ti / open-source [QualCoder](https://github.com/ccbogel/QualCoder)), and de-identification parallels spaCy / Presidio-style PII detection.

We use a small toy corpus to see each step clearly: 3 short "interviews" (int01–int03), one or two sentences each, deliberately embedded with names, email addresses, and phone numbers. Small scale makes it easy to keep track of offsets, quotes, and verification rates—details often overlooked.

In [1]:
import matplotlib
matplotlib.use("Agg")  # 无显示环境:图直接写文件

import json
import socialverse as sv
from socialverse import datasets as ds

## Loading the corpus

The minimum addressable unit in qualitative analysis is a "coding unit"—typically a paragraph. `build_corpus` first applies Unicode normalization (NFC) to each document, then partitions it into units carrying **stable `unit_id` and character offsets**. This `(doc_id, start, end)` offset is the foundation for later quote tracing: with it, any quote can be sliced back to the original text for verbatim comparison.

We first write the raw interviews to the research state's `sources` slot, then call `build_corpus`. The `manifest` summarizes how many units and characters each document yields; each unit's `unit_id` embeds the document identifier and character span.

In [2]:
st = sv.StudyState()
st.write("sources", "corpora", ds.load_corpus())  # 3 段短访谈,含姓名/邮箱/电话

sv.pp.build_corpus(st)

print("文档数:", len(st.corpus["documents"]), " · 编码单元数:", len(st.corpus["units"]))
st.corpus["manifest"]

文档数: 3  · 编码单元数: 3


,doc_id,n_units,n_chars
0,int01,1,184
1,int02,1,185
2,int03,1,177


Look at the first coding unit. Note that `unit_id` has the form `int01:0-184`—document identifier plus character range. That is what "addressable" means. At this point, the email address in the text is still plaintext; the next step removes it.

In [3]:
print(json.dumps(st.corpus["units"][0], ensure_ascii=False, indent=2))

{
  "unit_id": "int01:0-184",
  "doc_id": "int01",
  "start": 0,
  "end": 184,
  "text": "My name is Jane Doe and you can reach me at jane.doe@example.com. Honestly the workload here is crushing; I feel burned out most weeks, but my team supports me and that keeps me going."
}


## De-identification

Before any coding or sharing, remove PII. `redact_pii` uses deterministic regex to identify email addresses, phone numbers, long digit sequences, and other entities (if spaCy is installed, it also runs NER for person names), replacing each entity with a **stable pseudonym**—`[EMAIL_1]`, `[PHONE_1]`, etc. The same entity always maps to the same token across the entire corpus. It simultaneously generates a `crosswalk` (pseudonym → original text) for authorized users to re-identify if needed, and writes a compliance receipt `pii_status`.

Below we compare the same text segment from int01 before and after redaction, then print the receipt and crosswalk.

In [4]:
before = st.corpus["documents"]["int01"]

sv.pp.redact_pii(st)  # 原地擦洗 documents,并写 governance 回执

after = st.corpus["documents"]["int01"]
print("擦洗前:", before)
print("擦洗后:", after)

擦洗前: My name is Jane Doe and you can reach me at jane.doe@example.com. Honestly the workload here is crushing; I feel burned out most weeks, but my team supports me and that keeps me going.
擦洗后: My name is Jane Doe and you can reach me at [EMAIL_1]. Honestly the workload here is crushing; I feel burned out most weeks, but my team supports me and that keeps me going.


In [5]:
print("pii_status(合规回执):", st.governance["pii_status"])
print("\ncrosswalk(假名 → 原文,可逆再识别,须授权):")
print(json.dumps(st.governance["pii_crosswalk"], ensure_ascii=False, indent=2))

pii_status(合规回执): {'method': 'regex', 'crosswalk_size': 3, 'date': '2026-07-06'}

crosswalk(假名 → 原文,可逆再识别,须授权):
{
  "[EMAIL_1]": "jane.doe@example.com",
  "[PHONE_1]": "415-555-0132",
  "[EMAIL_2]": "li.wei@example.org"
}


Here lies a genuine trap worth pausing to understand. The `units` from the previous step were sliced from **pre-redaction** text, but `redact_pii` only changed `documents`. `[EMAIL_1]` is shorter than the original email `jane.doe@example.com`, so the character offsets in the redacted document have shifted. If we now attempt quote tracing, the system will use the old units' `(start, end)` to slice the **new** documents, and verbatim comparison will fail. In the next section we deliberately encounter this failure and repair it—and because the failure is accurately reported, that is where this pipeline's value lies.

## Thematic coding

This implements phases 2–4 of Braun & Clarke's six-stage process: apply a **coding lexicon** to each unit (mapping keywords to codes), record matching segments, and group codes into higher-order **themes** according to the researcher's judgment. Simultaneously build a code co-occurrence graph for later theme mapping.

`code_themes` handles coding and theming; `trace_quotes` verifies each matched segment against the source text and builds a quote index. We first run it using the **current (redacted but not re-cut)** units to observe what the quote verification rate looks like.

In [6]:
LEXICON = {
    "burnout":     ["burnout", "burned out", "crushing"],
    "support":     ["support", "belonging", "colleagues"],
    "autonomy":    ["autonomy", "flexibility"],
    "recognition": ["recognition", "morale"],
}
# 把 codes 归入更高阶主题(这一步是研究者的解释判断,不是自动的)
THEME_GROUPS = {
    "工作压力": ["burnout"],
    "组织支持": ["support", "recognition"],
    "工作条件": ["autonomy"],
}

sv.tl.code_themes(st, lexicon=LEXICON, themes=THEME_GROUPS)
sv.tl.trace_quotes(st)

cov1 = st.evidence["quote_index"]["coverage"]
print("第一版引语回校:verify_rate =", cov1["verify_rate"],
      f"({cov1['n_verified']}/{cov1['n_checkable']} 条逐字回校通过)")

第一版引语回校:verify_rate = 0.0 (0/7 条逐字回校通过)


`verify_rate = 0.0`—not a single quote verified. The cause is the trap from the previous section: units were sliced from pre-redaction text, and their offsets no longer align with the redacted documents. In many point-and-click qualitative software packages, this misalignment is silent; here, because tracing uses verbatim slicing to verify, it is caught immediately. The fix is straightforward: **after de-identification, re-cut units from the redacted documents**, ensuring coding units align perfectly with the final text, then re-code and re-trace.

In [7]:
# 用擦洗后的 documents 重切编码单元(覆盖旧的、偏移错位的 units)
sv.pp.build_corpus(st, data=st.corpus["documents"])

# 在对齐后的单元上重新编码 + 重新溯源
sv.tl.code_themes(st, lexicon=LEXICON, themes=THEME_GROUPS)
sv.tl.trace_quotes(st)

cov2 = st.evidence["quote_index"]["coverage"]
print("第二版引语回校:verify_rate =", cov2["verify_rate"],
      f"({cov2['n_verified']}/{cov2['n_checkable']} 条逐字回校通过)")
print("\n完整 coverage:")
print(json.dumps(cov2, ensure_ascii=False, indent=2))

第二版引语回校:verify_rate = 1.0 (7/7 条逐字回校通过)

完整 coverage:
{
  "n_segments": 7,
  "n_codes_with_quote": 4,
  "n_units_coded": 3,
  "n_units_total": 3,
  "unit_coverage": 1.0,
  "n_verified": 7,
  "n_checkable": 7,
  "verify_rate": 1.0
}


Now 100% verification passes, and every unit is coded at least once (`unit_coverage = 1.0`). This "de-identify-first" ordering guarantees: every unit entering the coding phase is PII-free and can be traced verbatim to its source.

### Codebook

The codebook is the qualitative-software equivalent of a "Code Book": for each code, how many times it appears, which theme it belongs to, and which keywords define it. Here it is a real `pandas.DataFrame`, sorted by frequency, ready to go into a manuscript appendix.

In [8]:
st.codes["codebook"]

,code,count,theme,def
0,autonomy,2,工作条件,"keywords: autonomy, flexibility"
1,burnout,2,工作压力,"keywords: burned out, burnout, crushing"
2,support,2,组织支持,"keywords: belonging, colleagues, support"
3,recognition,1,组织支持,"keywords: morale, recognition"


### Themes and supporting evidence

`themes` groups codes into researcher-named higher-order themes, recording which codes and units support each theme. `claim_evidence` further restructures this into "claim → support" form—"theme X is supported by N coding units"—statements that can go directly into the results section, each traceable back to source text.

In [9]:
print("themes(主题 → codes / n_segments / 支撑 unit_ids):")
print(json.dumps(st.codes["themes"], ensure_ascii=False, indent=2))

themes(主题 → codes / n_segments / 支撑 unit_ids):
{
  "工作压力": {
    "codes": [
      "burnout"
    ],
    "n_segments": 2,
    "unit_ids": [
      "int01:0-173",
      "int03:0-168"
    ]
  },
  "组织支持": {
    "codes": [
      "support",
      "recognition"
    ],
    "n_segments": 3,
    "unit_ids": [
      "int01:0-173",
      "int03:0-168",
      "int02:0-182"
    ]
  },
  "工作条件": {
    "codes": [
      "autonomy"
    ],
    "n_segments": 2,
    "unit_ids": [
      "int02:0-182",
      "int03:0-168"
    ]
  }
}


In [10]:
print("claim_evidence(每个主题的论断 ⇄ 支撑证据):")
print(json.dumps(st.evidence["claim_evidence"], ensure_ascii=False, indent=2))

claim_evidence(每个主题的论断 ⇄ 支撑证据):
{
  "工作压力": {
    "claim": "主题「工作压力」得到 2 个编码单元支撑",
    "codes": [
      "burnout"
    ],
    "support_unit_ids": [
      "int01:0-173",
      "int03:0-168"
    ],
    "n_support": 2
  },
  "组织支持": {
    "claim": "主题「组织支持」得到 3 个编码单元支撑",
    "codes": [
      "support",
      "recognition"
    ],
    "support_unit_ids": [
      "int01:0-173",
      "int03:0-168",
      "int02:0-182"
    ],
    "n_support": 3
  },
  "工作条件": {
    "claim": "主题「工作条件」得到 2 个编码单元支撑",
    "codes": [
      "autonomy"
    ],
    "support_unit_ids": [
      "int02:0-182",
      "int03:0-168"
    ],
    "n_support": 2
  }
}


## Quote tracing

This is the core deliverable of this notebook. For each theme, list **verbatim-verified** quotes along with their precise provenance stamp `(doc_id, start, end)`. `verified=True` indicates that slicing the final document at that offset produces text that matches the quote exactly as coded. Below we print by theme.

In [11]:
entries = st.evidence["quote_index"]["entries"]

def quotes_for_theme(theme_name):
    """返回某主题下所有已溯源的引语条目。"""
    codes = set(st.codes["themes"][theme_name]["codes"])
    return [e for e in entries if e["code"] in codes]

for theme in st.codes["themes"]:
    print(f"■ 主题「{theme}」")
    for e in quotes_for_theme(theme):
        stamp = f"{e['doc_id']}[{e['start']}:{e['end']}]"
        flag = "✓已回校" if e["verified"] else "✗未回校"
        print(f"   [{e['code']:<11}] {stamp:<16} {flag}")
        print(f"      引语: {e['quote'][:88]}")
    print()

■ 主题「工作压力」
   [burnout    ] int01[0:173]     ✓已回校
      引语: My name is Jane Doe and you can reach me at [EMAIL_1]. Honestly the workload here is cru
   [burnout    ] int03[0:168]     ✓已回校
      引语: Contact: [EMAIL_2]. What I value most is flexibility and the support of colleagues. Burn

■ 主题「组织支持」
   [support    ] int01[0:173]     ✓已回校
      引语: My name is Jane Doe and you can reach me at [EMAIL_1]. Honestly the workload here is cru
   [recognition] int02[0:182]     ✓已回校
      引语: I am Robert Smith. Call me on [PHONE_1]. The pay is fine but the lack of autonomy frustr
   [support    ] int03[0:168]     ✓已回校
      引语: Contact: [EMAIL_2]. What I value most is flexibility and the support of colleagues. Burn

■ 主题「工作条件」
   [autonomy   ] int02[0:182]     ✓已回校
      引语: I am Robert Smith. Call me on [PHONE_1]. The pay is fine but the lack of autonomy frustr
   [autonomy   ] int03[0:168]     ✓已回校
      引语: Contact: [EMAIL_2]. What I value most is flexibility and the support of colleagues. Burn

Run one more orphan audit: are there codes with no quotes, or units with no codes? Both are empty lists, confirming complete coding coverage.

In [12]:
print("孤儿审计:", json.dumps(st.evidence["quote_index"]["orphans"], ensure_ascii=False))

孤儿审计: {"codes_without_quote": [], "units_without_code": []}


## Reflexive memo

What distinguishes reflexive thematic analysis from "mechanical coding" is that the researcher must disclose their **positionality** and **interpretive trajectory**—typically scattered across private notes and difficult to audit. `reflexive_memo` structures this into an auditable protocol: a three-axis positionality statement (social location / field relation / stakes), a four-part journal entry per theme (observation / response / bias / adjustment), and an explicit **AI vs. human** attribution—which interpretations are automated and which are the researcher's judgment. The three axes require the researcher to fill in directly; once completed, the ethics-statement status changes from "pending" to `declared`.

In [13]:
sv.tl.reflexive_memo(
    st,
    researcher="FZ",
    positionality={
        "social_location": "组织社会学训练、非该行业从业者的外部研究者。",
        "field_relation": "与受访者无雇佣或师生关系,访谈为一次性、自愿参与。",
        "stakes": "研究不涉及商业委托,发现对研究者无直接利害;对受访机构可能敏感。",
    },
    authorship={
        "ai": ["按词典的自动编码命中", "code 共现主题地图的计算"],
        "human": ["主题命名与解释", "立场声明", "最终 claim–evidence 判断"],
    },
)

memo = st.evidence["provenance"]
print("立场声明(三轴,研究者亲填):")
print(json.dumps(memo["positionality"], ensure_ascii=False, indent=2))

立场声明(三轴,研究者亲填):
{
  "social_location": "组织社会学训练、非该行业从业者的外部研究者。",
  "field_relation": "与受访者无雇佣或师生关系,访谈为一次性、自愿参与。",
  "stakes": "研究不涉及商业委托,发现对研究者无直接利害;对受访机构可能敏感。"
}


Each theme automatically gets a four-part journal-entry skeleton (observation is filled, response / bias / adjustment left for the researcher), plus an AI-versus-human attribution statement that should go into the methods and acknowledgments sections of the paper.

In [14]:
first_theme = next(iter(memo["log"]))
print(f"主题「{first_theme}」的四段反身日志:")
print(json.dumps(memo["log"][first_theme], ensure_ascii=False, indent=2))

主题「工作压力」的四段反身日志:
{
  "observation": "主题「工作压力」由 1 个编码、2 个单元构成。",
  "reaction": "研究者对「工作压力」的直觉反应与初始解读 — 待研究者补写。",
  "bias": "可能影响对「工作压力」判读的先见/立场偏差 — 待研究者识别。",
  "adjustment": "据反身审视对「工作压力」编码/解读所做的调整 — 待研究者记录。"
}


In [15]:
print("解释归属 AI vs 人类:")
print(json.dumps(memo["interpretation_authorship"], ensure_ascii=False, indent=2))
print("\nethics(反身性声明合规状态):")
print(json.dumps(st.governance["ethics"], ensure_ascii=False, indent=2))

解释归属 AI vs 人类:
{
  "ai": [
    "按词典的自动编码命中",
    "code 共现主题地图的计算"
  ],
  "human": [
    "主题命名与解释",
    "立场声明",
    "最终 claim–evidence 判断"
  ],
  "policy": "reflexive TA: 生成式辅助编码与结构脚手架由 AI 承担,解释与主题命名归研究者;须在方法与致谢中披露。"
}

ethics(反身性声明合规状态):
{
  "reflexivity_declared": true,
  "positionality_complete": true,
  "ai_use_disclosed": true,
  "log_entries": 3,
  "status": "declared",
  "note": "反身性声明已提供"
}


## Theme map

Finally, visualize the code co-occurrence structure as a network: nodes are codes (sized by weighted degree), edges represent co-occurrence frequency, color encodes theme membership. This is the reader-facing snapshot of theme structure. `theme_map` uses deterministic layout (`seed=0`), reads the graph directly from the research state, and saves it as PNG to the same directory.

In [16]:
sv.pl.theme_map(st, out="fig_thememap.png", title="访谈主题共现网络")

fig_meta = st.artifacts["figures"]["theme_map"]
print("已保存图:", fig_meta["note"], "· dpi:", fig_meta["dpi"])
print("\ntheme_map 邻接结构(共现次数):")
print(json.dumps(st.codes["theme_map"], ensure_ascii=False, indent=2))

已保存图: 4 节点 / 4 边 主题共现图 · dpi: 200

theme_map 邻接结构(共现次数):
{
  "nodes": [
    "burnout",
    "support",
    "autonomy",
    "recognition"
  ],
  "adjacency": {
    "burnout": {
      "support": 2,
      "autonomy": 1
    },
    "support": {
      "burnout": 2,
      "autonomy": 1
    },
    "autonomy": {
      "recognition": 1,
      "burnout": 1,
      "support": 1
    },
    "recognition": {
      "autonomy": 1
    }
  },
  "n_edges": 4,
  "density": 0.6667
}


![Theme co-occurrence network](fig_thememap.png)

`burnout` and `support` co-occur most densely (the same respondent often mentions peer support while discussing stress), while `autonomy` and `recognition` each link to neighboring themes—this map lays out how themes interweave.

## Reproducible audit trail

Unlike a typical qualitative-coding script, `socialverse` preserves an extra artifact: as the pipeline runs, the research state automatically accumulates a provenance ledger, recording which function each step called, which slots it consumed, and which slots it produced. In qualitative research, "which step, which dataset, which respondent's exact words is this conclusion from" matters as much as the conclusion itself—this ledger makes it auditable and reproducible.

In [17]:
print(st.summary())

StudyState
  sources: corpora
  corpus: documents, units, manifest
  codes: codebook, segments, themes, theme_map
  evidence: provenance, claim_evidence, quote_index
  governance: pii_status, pii_crosswalk, ethics
  artifacts: tables, figures
  provenance: 9 step(s)


The ledger shows we ran coding and tracing **twice**: steps 3–4 hit the pitfall, steps 5–7 are the repaired version after re-cutting. Any final quote can be traced back through the ledger: theme → supporting unit → character offset → de-identification receipt → raw corpus.

In [18]:
for r in st.provenance:
    req = ", ".join(f"{s}[{','.join(k)}]" for s, k in r["requires"].items()) or "∅"
    pro = ", ".join(f"{s}[{','.join(k)}]" for s, k in r["produces"].items()) or "∅"
    print(f"step {r['step']}: {sv.utils._friendly(r['function'])}")
    print(f"   requires {req}")
    print(f"   produces {pro}")

step 1: sv.pp.build_corpus
   requires sources[corpora]
   produces corpus[documents,units,manifest], evidence[provenance]
step 2: sv.pp.redact_pii
   requires corpus[documents]
   produces corpus[documents], governance[pii_status]
step 3: sv.tl.code_themes
   requires corpus[units]
   produces codes[codebook,segments,themes,theme_map], evidence[claim_evidence], artifacts[tables]
step 4: sv.tl.trace_quotes
   requires corpus[units], codes[segments]
   produces evidence[quote_index]
step 5: sv.pp.build_corpus
   requires sources[corpora]
   produces corpus[documents,units,manifest], evidence[provenance]
step 6: sv.tl.code_themes
   requires corpus[units]
   produces codes[codebook,segments,themes,theme_map], evidence[claim_evidence], artifacts[tables]
step 7: sv.tl.trace_quotes
   requires corpus[units], codes[segments]
   produces evidence[quote_index]
step 8: sv.tl.reflexive_memo
   requires codes[themes], corpus[units]
   produces evidence[provenance], governance[ethics]
step 9: sv.p

## Summary

We have completed a full qualitative-coding pipeline: load corpus → de-identify → code themes → trace quotes → reflexive memo → theme map. It mirrors the **CAQDAS** (NVivo / ATLAS.ti / QualCoder) code–retrieve–visualize workflow; the methodological framework is Braun & Clarke's six-stage reflexive thematic analysis, and de-identification parallels spaCy / Presidio.

Compared to point-and-click software, this approach adds two things: de-identification is a **default-first, reversible-crosswalk** compliance gate, and quote tracing **verbatim-verifies** every quote against source character offsets—verification failures are accurately reported (as in this notebook's offset-mismatch trap) rather than silently swallowed. The next tutorial, [06_text_philology](06_text_philology.ipynb), takes text processing deeper: OCR, textual collation, and TEI markup.